In [ ]:
!pip install transformers[torch] datasets evaluate

In [ ]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate

# 1. Load your specific file
df = pd.read_csv('dark-patterns-v2.csv')

# 2. Cleaning (Matches the processing we just did)
df = df.dropna(subset=['Pattern String'])
unique_labels = df['Pattern Category'].unique().tolist()
label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for label, i in label2id.items()}
df['label'] = df['Pattern Category'].map(label2id)
df['text'] = df['Pattern String'].astype(str)

# 3. Prepare Dataset
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
dataset = Dataset.from_pandas(df[['text', 'label']])

def tokenize_func(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_ds = dataset.map(tokenize_func, batched=True).train_test_split(test_size=0.2)

# 4. Model Setup
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(unique_labels),
    id2label=id2label,
    label2id=label2id
)

# 5. Training (Run for 3 epochs for the progress report results)
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",      # Add this to save checkpoints
    logging_steps=10,
    num_train_epochs=10,
    per_device_train_batch_size=16,
    weight_decay=0.01,
    load_best_model_at_end=True # Recommended for better accuracy
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
)

trainer.train()

In [ ]:
from google.colab import files

# Zip the results folder so it's easy to download
!zip -r model_results.zip ./results

# Download to your local 'Downloads' folder
files.download('model_results.zip')